# Memorization Comparison: Masked vs Unmasked (1B & 7B)

Compare memorization statistics across different training configurations:
- **1B Masked** (`Train_95frozen_FullSubset`)
- **1B Unmasked** (`Train_UnMask_FullSubset`)
- **7B Masked** (`7B_Train_95frozen_FullSubset`)
- **7B Unmasked** (`7B_UnMask_FullSubset`)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Color scheme matching runs_performance_analysis.ipynb
COLOR_MASKED = '#5964FF'
COLOR_UNMASKED = '#662E7D'
CONFIG_COLORS = {
    '1B Masked': COLOR_MASKED, '1B Unmasked': COLOR_UNMASKED,
    '7B Masked': COLOR_MASKED, '7B Unmasked': COLOR_UNMASKED,
}

LEGEND_SIZE = 26
TICK_SIZE = 30
TITLE_SIZE = 35
LABEL_SIZE = 35
BAR_LABEL_SIZE = 24

In [ ]:
# --- Configuration: paths to memo_results.csv files ---
import os
BASE = os.path.join(os.environ['HOME'], 'OLMoBenchOutputs', 'saves')

configs = {
    '1B Masked':   f'{BASE}/Train_95frozen_FullSubset/memo_results.csv',
    '1B Unmasked': f'{BASE}/Train_UnMask_FullSubset/memo_results.csv',
    '7B Masked':   f'{BASE}/7B_Train_95frozen_FullSubset/memo_results.csv',
    '7B Unmasked': f'{BASE}/7B_UnMask_FullSubset/memo_results.csv',
}

FIELDS = ['Birth City', 'Email Address', 'Phone Number', "Driver's License"]

In [ ]:
# Load all datasets
dfs = {}
for name, path in configs.items():
    try:
        df = pd.read_csv(path)
        df['Config'] = name
        dfs[name] = df
        print(f"{name}: {len(df):,} rows, {df['UniqueID'].nunique()} unique people")
    except FileNotFoundError:
        print(f"{name}: FILE NOT FOUND at {path}")

## 1. Exact Match Comparison

In [ ]:
# Per-person exact match rate (at least one correct answer per person per field)
records = []
for name, df in dfs.items():
    for field in FIELDS:
        curr = df[df['Field'] == field]
        if len(curr) == 0:
            continue
        per_person = curr.groupby('UniqueID')['Is_Correct'].any()
        rate = per_person.mean() * 100
        n_correct = per_person.sum()
        n_total = len(per_person)
        records.append({
            'Config': name,
            'Field': field,
            'Exact Match %': round(rate, 2),
            'Correct': n_correct,
            'Total': n_total,
        })

em_df = pd.DataFrame(records)
em_pivot = em_df.pivot_table(index='Field', columns='Config', values='Exact Match %')
em_pivot = em_pivot[['1B Masked', '1B Unmasked', '7B Masked', '7B Unmasked']]
em_pivot

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os
from matplotlib import rcParams
import textwrap

ASSETS_DIR = os.path.join((os.getcwd() if os.path.basename(os.getcwd()) == 'notebooks' else os.path.join(os.getcwd(), 'notebooks')), 'assets', 'imgs')
os.makedirs(ASSETS_DIR, exist_ok=True)

# 1. Global Aesthetic Settings
rcParams['font.family'] = 'monospace'
rcParams['font.monospace'] = ['Inconsolata', 'Consolas', 'DejaVu Sans Mono']
rcParams['font.style'] = 'normal'
rcParams['font.weight'] = 'bold'

# 2. Font Sizes (matching memorization_comparison_1B_7B style)
TITLE_SIZE = 56
LABEL_SIZE = 48
TICK_SIZE = 40
TRUE_BLACK = '#000000'

sizes = {'1B': 'OLMo2 1B', '7B': 'OLMo3 7B'}

# 3. Setup Figure
fig, axes = plt.subplots(1, 2, figsize=(40, 12), sharey=True)

for i, (ax, size) in enumerate(zip(axes, sizes.keys())):
    # Data processing
    subset = em_df[em_df['Config'].str.startswith(size)]
    pivot = subset.pivot(index='Field', columns='Config', values='Exact Match %')
    cols = [f'{size} Masked', f'{size} Unmasked']

    n_groups = len(pivot.index)
    n_bars = len(cols)
    x = np.arange(n_groups) * 1.3
    width = 0.4
    colors = [COLOR_MASKED, COLOR_UNMASKED]

    # 4. Manual Bar Plotting
    for j, col in enumerate(cols):
        offset = (j - (n_bars - 1) / 2) * width
        rects = ax.bar(
            x + offset,
            pivot[col],
            width,
            color=colors[j],
            edgecolor='black',
            linewidth=2.0
        )

    # 5. Styling (matching 1B_7B chart)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(2)
    ax.spines['bottom'].set_linewidth(2)
    ax.spines['left'].set_color(TRUE_BLACK)
    ax.spines['bottom'].set_color(TRUE_BLACK)

    ax.set_title(sizes[size], fontsize=TITLE_SIZE, fontweight='bold')

    if i == 0:
        ax.set_ylabel('Extraction Rate %',
                      fontsize=LABEL_SIZE, fontweight='bold', labelpad=40)
    else:
        ax.set_ylabel('')

    ax.set_xlabel('')
    ax.set_ylim(0, 100)
    ax.set_yticks(range(0, 100, 20))
    ax.set_yticklabels(range(0, 100, 20))

    # X-Axis: wrapped labels, centered
    ax.set_xticks(x)
    wrapped_labels = [textwrap.fill(str(label), width=8) for label in pivot.index]
    ax.set_xticklabels(wrapped_labels,
                       fontsize=TICK_SIZE,
                       fontweight='bold',
                       ha='center')

    # Tick styling
    ax.tick_params(axis='y', labelsize=TICK_SIZE, width=2, length=12, colors=TRUE_BLACK)
    ax.tick_params(axis='x', width=2, length=12, pad=15)
    ax.yaxis.set_tick_params(which='both', left=True)
    ax.xaxis.set_tick_params(which='both', bottom=True)

    ax.grid(False, axis='x')
    ax.grid(axis='y', linestyle='--', alpha=0.3, linewidth=3, zorder=0)

plt.tight_layout()
out_path = os.path.join(ASSETS_DIR, 'memorization_comparison.pdf')
fig.savefig(out_path, format='pdf', bbox_inches='tight')
print(f"Saved to {out_path}")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os
from matplotlib import rcParams
import textwrap

ASSETS_DIR = os.path.join((os.getcwd() if os.path.basename(os.getcwd()) == 'notebooks' else os.path.join(os.getcwd(), 'notebooks')), 'assets', 'imgs')
os.makedirs(ASSETS_DIR, exist_ok=True)

rcParams['font.family'] = 'monospace'
rcParams['font.monospace'] = ['Inconsolata', 'Consolas', 'DejaVu Sans Mono']
rcParams['font.style'] = 'normal'
rcParams['font.weight'] = 'bold'

LABEL_SIZE = 72
TICK_SIZE = 60
LEGEND_SIZE = 52
TRUE_BLACK = '#000000'

pivot = em_df.pivot(index='Field', columns='Config', values='Exact Match %')
cols = ['1B Masked', '7B Masked']

# 0.33/20 = 0.65/39.4 = 0.0165 (equal scale factor), H=18 same as perf
fig, ax = plt.subplots(figsize=(20, 18))

n_groups = len(pivot.index)
x = np.arange(n_groups) * 1.2
width = 0.35
COLOR_1B = '#B2B8FF'
COLOR_7B = '#2A32B2'
colors = [COLOR_1B, COLOR_7B]

for j, col in enumerate(cols):
    offset = (j - 0.5) * width
    ax.bar(x + offset, pivot[col], width,
           label=col.replace(' Masked', ''),
           color=colors[j], edgecolor='black', linewidth=2.5)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_linewidth(3)
ax.spines['bottom'].set_linewidth(3)
ax.spines['left'].set_color(TRUE_BLACK)
ax.spines['bottom'].set_color(TRUE_BLACK)

ax.set_ylabel('Extraction Rate %',
              fontsize=LABEL_SIZE, fontweight='bold', labelpad=40)
ax.set_ylim(0, 100)
ax.set_yticks(range(0, 101, 20))

ax.set_xticks(x)
wrapped_labels = [textwrap.fill(str(label), width=8) for label in pivot.index]
ax.set_xticklabels(wrapped_labels, fontsize=TICK_SIZE, fontweight='bold', ha='center')

ax.tick_params(axis='y', labelsize=TICK_SIZE, width=3, length=14, colors=TRUE_BLACK)
ax.tick_params(axis='x', width=3, length=14, pad=20)

ax.grid(False, axis='x')
ax.grid(axis='y', linestyle='--', alpha=0.3, linewidth=2.5, zorder=0)

ax.legend(fontsize=LEGEND_SIZE, frameon=False, loc='upper left')

# left=0.18 → 3.6in for ylabel+ticks; bottom=0.14 shared with perf
fig.subplots_adjust(left=0.18, right=0.97, bottom=0.14, top=0.97)

out_path = os.path.join(ASSETS_DIR, 'memorization_comparison_1B_7B.pdf')
fig.savefig(out_path, format='pdf')
print(f"Saved to {out_path}")
plt.show()